# Assignment 2 – Data Preprocessing & Feature Engineering
## Snitch Fashion Sales Dataset

**Tasks covered:**
1. Identify data quality issues
2. Apply one missing value strategy with justification
3. Detect and handle outliers using IQR
4. Normalize numerical features (Min-Max & Z-score)
5. Apply PCA and interpret explained variance

## 0. Import Libraries & Recreate the Cleaned Dataset
Since Assignment 1 produced the cleaned CSV, we rebuild it here so the notebook is self-contained.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')

# ── Load raw dataset (same file used in Assignment 1) ──────────────────────
DATA_PATH = Path('Snitch_Fashion_Sales_Uncleaned.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('/mnt/data/Assignment1/Snitch_Fashion_Sales_Uncleaned.csv')

df_raw = pd.read_csv(DATA_PATH)
print('Raw dataset shape:', df_raw.shape)
df_raw.head()

In [ ]:
# ── Reproduce the same cleaning pipeline from Assignment 1 ────────────────
cleaned = df_raw.copy()
cleaned.insert(0, 'Record_ID', range(1, len(cleaned) + 1))

for col in ['Customer_Name', 'Product_Category', 'Product_Name', 'City', 'Segment']:
    cleaned[col] = cleaned[col].astype('string').str.strip()

city_map = {'Bangalore': 'Bengaluru', 'bengaluru': 'Bengaluru',
            'Hyd': 'Hyderabad', 'hyderbad': 'Hyderabad'}
cleaned['City'] = cleaned['City'].replace(city_map)

cleaned['Segment'] = cleaned['Segment'].astype('string').str.upper()
cleaned.loc[~cleaned['Segment'].isin(['B2B', 'B2C']), 'Segment'] = pd.NA

dates = pd.to_datetime(cleaned['Order_Date'], errors='coerce', dayfirst=True, format='mixed')

cleaned.loc[cleaned['Units_Sold'] <= 0, 'Units_Sold'] = np.nan
cleaned.loc[cleaned['Unit_Price'] <= 0, 'Unit_Price'] = np.nan
cleaned.loc[(cleaned['Discount_%'] < 0) | (cleaned['Discount_%'] > 1), 'Discount_%'] = np.nan

def fill_group_median(frame, column, groups):
    s = frame[column].copy()
    for g in groups:
        s = s.fillna(frame.groupby(g)[column].transform('median'))
    return s.fillna(frame[column].median())

seg_mode = cleaned.groupby('Customer_Name')['Segment'].agg(
    lambda x: x.dropna().mode().iloc[0] if not x.dropna().mode().empty else pd.NA)
cleaned['Segment'] = cleaned['Segment'].fillna(cleaned['Customer_Name'].map(seg_mode))
cleaned['Segment'] = cleaned['Segment'].fillna(cleaned['Segment'].mode(dropna=True).iloc[0])

cleaned['Unit_Price']  = fill_group_median(cleaned, 'Unit_Price',  ['Product_Name', 'Product_Category', 'City'])
cleaned['Units_Sold']  = fill_group_median(cleaned, 'Units_Sold',  ['Product_Name', 'Product_Category', 'Segment'])
cleaned['Discount_%']  = fill_group_median(cleaned, 'Discount_%',  ['Product_Category', 'Segment'])

cleaned['Units_Sold']  = cleaned['Units_Sold'].round().clip(lower=1).astype(int)
cleaned['Unit_Price']  = cleaned['Unit_Price'].round(2)
cleaned['Discount_%']  = cleaned['Discount_%'].clip(lower=0, upper=1).round(2)

city_med = cleaned.assign(_d=dates).groupby('City')['_d'].transform(
    lambda s: s.dropna().sort_values().iloc[len(s.dropna()) // 2] if len(s.dropna()) else pd.NaT)
dates = dates.fillna(city_med)
dates = dates.fillna(dates.dropna().sort_values().iloc[len(dates.dropna()) // 2])

cleaned['Order_Date']  = dates.dt.strftime('%Y-%m-%d')
cleaned['Order_Year']  = dates.dt.year.astype(int)
cleaned['Order_Month'] = dates.dt.month.astype(int)

cleaned['Discount']      = cleaned['Discount_%']
cleaned['Sales_Amount']  = (cleaned['Units_Sold'] * cleaned['Unit_Price'] * (1 - cleaned['Discount'])).round(2)
cleaned['Profit_Flag']   = np.where(cleaned['Profit'] > cleaned['Sales_Amount'], 'Review', 'OK')
cleaned.loc[cleaned['Profit'] < 0, 'Profit_Flag'] = 'Negative'

cleaned = cleaned[['Record_ID','Order_ID','Order_ID_Duplicate_Flag','Customer_Name',
                   'Product_Category','Product_Name','Units_Sold','Unit_Price','Discount',
                   'Sales_Amount','Order_Date','Order_Year','Order_Month','City','Segment',
                   'Profit','Profit_Flag']]

print('Cleaned dataset shape:', cleaned.shape)
cleaned.head()

---
## Task 1 – Identify Data Quality Issues

In [ ]:
# ── 1a. Missing values in the RAW dataset ──────────────────────────────────
missing = pd.DataFrame({
    'Missing Count': df_raw.isna().sum(),
    'Missing %': (df_raw.isna().mean() * 100).round(2)
}).sort_values('Missing Count', ascending=False)

print('=== Missing Values (Raw Dataset) ===')
print(missing[missing['Missing Count'] > 0])

In [ ]:
# ── 1b. Duplicate rows and duplicate Order_IDs ──────────────────────────────
print('Exact duplicate rows    :', df_raw.duplicated().sum())
print('Duplicate Order_ID rows :', df_raw['Order_ID'].duplicated().sum())

In [ ]:
# ── 1c. Invalid numeric values ──────────────────────────────────────────────
print('Units_Sold <= 0 :', ((df_raw['Units_Sold'] <= 0).fillna(False)).sum())
print('Discount_%  < 0 or > 1 :',
      (((df_raw['Discount_%'] < 0) | (df_raw['Discount_%'] > 1)).fillna(False)).sum())
print('Sales_Amount = 0 :', (df_raw['Sales_Amount'] == 0).sum())
print('Sales_Amount < 0 :', (df_raw['Sales_Amount'] < 0).sum())

In [ ]:
# ── 1d. Inconsistent categorical values ──────────────────────────────────────
print('Unique City values (raw):',   sorted(df_raw['City'].dropna().unique()))
print('\nUnique Segment values (raw):', sorted(df_raw['Segment'].dropna().unique()))

In [ ]:
# ── 1e. Mixed date formats ──────────────────────────────────────────────────
print('Sample Order_Date values (raw):')
print(df_raw['Order_Date'].dropna().sample(10, random_state=42).values)

In [ ]:
# ── 1f. Summary visualization ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
cols_with_missing = missing[missing['Missing Count'] > 0]
ax.bar(cols_with_missing.index, cols_with_missing['Missing %'], color='steelblue')
ax.set_title('Missing Values (%) – Raw Dataset')
ax.set_ylabel('Missing %')
ax.set_xlabel('Column')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("""
Summary of Data Quality Issues Found:
1. Missing values in Units_Sold, Unit_Price, Discount_%, Order_Date, Segment
2. Duplicate Order_ID entries (same order recorded more than once)
3. Invalid numeric values: Units_Sold <= 0, Discount_% outside [0, 1]
4. Inconsistent City names: 'Bangalore'/'bengaluru' vs 'Bengaluru', 'Hyd'/'hyderbad' vs 'Hyderabad'
5. Inconsistent Segment values: mixed case and non-standard labels
6. Mixed date formats in Order_Date (DD/MM/YYYY, YYYY-MM-DD, etc.)
7. Sales_Amount of zero or negative in some rows
""")

---
## Task 2 – Apply One Missing Value Strategy and Explain Why

**Strategy chosen: Group-based Median Imputation**

**Why this strategy?**
- The numerical columns (`Units_Sold`, `Unit_Price`, `Discount_%`) are skewed and contain outliers, so the **median** is more robust than the mean.
- Values within the same product group or city tend to be similar, so **grouping by `Product_Name`, `Product_Category`, and `City`** gives a more accurate estimate than a global median.
- Dropping rows was not suitable because the missing rate in some columns is high enough to cause significant data loss.
- A constant fill (e.g., 0) would distort distributions and be misleading.

**Implementation:** For each missing cell, imputation falls through a hierarchy of groups (most specific → least specific → global median), ensuring every row receives a plausible value.

In [ ]:
# ── Demonstrate on a working copy starting from the raw data ────────────────
df2 = df_raw.copy()

# Mark invalid numerics as NaN first (same as Assignment 1 cleaning)
df2.loc[df2['Units_Sold'] <= 0, 'Units_Sold'] = np.nan
df2.loc[df2['Unit_Price'] <= 0, 'Unit_Price'] = np.nan
df2.loc[(df2['Discount_%'] < 0) | (df2['Discount_%'] > 1), 'Discount_%'] = np.nan

# Strip text columns so groupby keys are consistent
for col in ['Product_Name', 'Product_Category', 'City', 'Segment']:
    df2[col] = df2[col].astype('string').str.strip()

missing_before = df2[['Units_Sold','Unit_Price','Discount_%']].isna().sum()
print('Missing values BEFORE imputation:')
print(missing_before)

# ── Apply group-based median imputation ────────────────────────────────────
def fill_group_median(frame, column, groups):
    """Fill NaN values using group medians, falling back to global median."""
    s = frame[column].copy()
    for g in groups:
        s = s.fillna(frame.groupby(g)[column].transform('median'))
    return s.fillna(frame[column].median())

df2['Unit_Price']  = fill_group_median(df2, 'Unit_Price',  ['Product_Name', 'Product_Category', 'City'])
df2['Units_Sold']  = fill_group_median(df2, 'Units_Sold',  ['Product_Name', 'Product_Category', 'Segment'])
df2['Discount_%']  = fill_group_median(df2, 'Discount_%',  ['Product_Category', 'Segment'])

missing_after = df2[['Units_Sold','Unit_Price','Discount_%']].isna().sum()
print('\nMissing values AFTER imputation:')
print(missing_after)

In [ ]:
# ── Visualize the effect of imputation on Units_Sold ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_raw['Units_Sold'].dropna(), bins=30, color='tomato', edgecolor='white')
axes[0].set_title('Units_Sold – Before Imputation (non-null only)')
axes[0].set_xlabel('Units Sold')
axes[0].set_ylabel('Frequency')

axes[1].hist(df2['Units_Sold'], bins=30, color='steelblue', edgecolor='white')
axes[1].set_title('Units_Sold – After Group-Median Imputation')
axes[1].set_xlabel('Units Sold')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

print("""
Interpretation:
The distribution shape is largely preserved after imputation.
Group-based median keeps the imputed values realistic within each product and city context,
avoiding the artificial spike at a single global value that simple mean/median fill would produce.
""")

---
## Task 3 – Detect and Handle Outliers Using IQR

The IQR method defines outliers as values below **Q1 − 1.5 × IQR** or above **Q3 + 1.5 × IQR**.

In [ ]:
# Work on the cleaned dataset from Assignment 1
NUM_COLS = ['Units_Sold', 'Unit_Price', 'Discount', 'Sales_Amount', 'Profit']

df3 = cleaned[NUM_COLS].copy()

# ── Compute IQR bounds ──────────────────────────────────────────────────────
Q1  = df3.quantile(0.25)
Q3  = df3.quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outlier_counts = ((df3 < lower) | (df3 > upper)).sum()

bounds = pd.DataFrame({'Q1': Q1, 'Q3': Q3, 'IQR': IQR,
                       'Lower Bound': lower, 'Upper Bound': upper,
                       'Outlier Count': outlier_counts})
print('IQR Outlier Analysis:')
print(bounds.to_string())

In [ ]:
# ── Visualize outliers with box plots BEFORE capping ───────────────────────
fig, axes = plt.subplots(1, len(NUM_COLS), figsize=(16, 5))
for ax, col in zip(axes, NUM_COLS):
    ax.boxplot(df3[col].dropna(), vert=True, patch_artist=True,
               boxprops=dict(facecolor='lightblue'))
    ax.set_title(col)
    ax.set_xticks([])
fig.suptitle('Box Plots – Before Outlier Handling', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Handle outliers: Winsorization (capping at IQR bounds) ─────────────────
# Winsorization is preferred over removal because:
#   - it retains all rows (no data loss)
#   - extreme values are capped at a meaningful boundary, not deleted
#   - the distribution shape is preserved while reducing the influence of extremes

df3_capped = df3.copy()
for col in NUM_COLS:
    df3_capped[col] = df3_capped[col].clip(lower=lower[col], upper=upper[col])

# ── Box plots AFTER capping ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(NUM_COLS), figsize=(16, 5))
for ax, col in zip(axes, NUM_COLS):
    ax.boxplot(df3_capped[col], vert=True, patch_artist=True,
               boxprops=dict(facecolor='lightgreen'))
    ax.set_title(col)
    ax.set_xticks([])
fig.suptitle('Box Plots – After Outlier Capping (Winsorization)', fontsize=13)
plt.tight_layout()
plt.show()

print('Rows before:', len(df3))
print('Rows after :', len(df3_capped), '(no rows removed – capping used)')

In [ ]:
# ── Summary statistics before vs after ─────────────────────────────────────
compare = pd.concat([
    df3.describe().loc[['mean','std','min','max']].rename(index=lambda x: 'before_' + x),
    df3_capped.describe().loc[['mean','std','min','max']].rename(index=lambda x: 'after_' + x)
])
print('\nStatistics Before vs After Capping:')
print(compare.to_string())

---
## Task 4 – Normalize Numerical Features (Min-Max & Z-score)

In [ ]:
# ── Min-Max Normalization ───────────────────────────────────────────────────
# Scales every feature to [0, 1].
# Useful when the algorithm requires bounded inputs (e.g., neural networks, KNN).

minmax_scaler = MinMaxScaler()
df_minmax = pd.DataFrame(
    minmax_scaler.fit_transform(df3_capped),
    columns=[c + '_minmax' for c in NUM_COLS]
)

print('Min-Max Normalized – first 5 rows:')
print(df_minmax.head().to_string())
print('\nMin values after scaling (should all be 0.0):')
print(df_minmax.min().values)
print('Max values after scaling (should all be 1.0):')
print(df_minmax.max().values)

In [ ]:
# ── Z-score Standardization ────────────────────────────────────────────────
# Transforms features to mean=0 and std=1.
# Useful for algorithms that assume normally distributed inputs (e.g., PCA, SVM, linear regression).

zscore_scaler = StandardScaler()
df_zscore = pd.DataFrame(
    zscore_scaler.fit_transform(df3_capped),
    columns=[c + '_zscore' for c in NUM_COLS]
)

print('Z-score Standardized – first 5 rows:')
print(df_zscore.head().to_string())
print('\nMean after standardization (should be ~0):')
print(df_zscore.mean().round(6).values)
print('Std after standardization (should be ~1):')
print(df_zscore.std().round(6).values)

In [ ]:
# ── Side-by-side distribution comparison ───────────────────────────────────
fig, axes = plt.subplots(3, len(NUM_COLS), figsize=(18, 10))

for i, col in enumerate(NUM_COLS):
    # Original
    axes[0, i].hist(df3_capped[col], bins=30, color='steelblue', edgecolor='white')
    axes[0, i].set_title(f'{col}\n(Original)')
    axes[0, i].set_ylabel('Frequency' if i == 0 else '')

    # Min-Max
    axes[1, i].hist(df_minmax[col + '_minmax'], bins=30, color='mediumseagreen', edgecolor='white')
    axes[1, i].set_title(f'{col}\n(Min-Max)')
    axes[1, i].set_ylabel('Frequency' if i == 0 else '')

    # Z-score
    axes[2, i].hist(df_zscore[col + '_zscore'], bins=30, color='darkorange', edgecolor='white')
    axes[2, i].set_title(f'{col}\n(Z-score)')
    axes[2, i].set_ylabel('Frequency' if i == 0 else '')

fig.suptitle('Feature Distributions: Original vs Min-Max vs Z-score', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print("""
Key Observations:
- Min-Max rescales values to [0, 1] while PRESERVING the shape of the distribution.
- Z-score shifts the center to 0 and scales spread to 1 while also PRESERVING the shape.
- Neither method changes the underlying distribution type (skewness is retained).
- Min-Max is sensitive to outliers; Z-score is more robust because it uses mean and std.
""")

---
## Task 5 – Apply PCA and Interpret Explained Variance

In [ ]:
# PCA is performed on Z-score standardized data (mean=0, std=1).
# Standardization is required before PCA so that features with larger scales
# do not dominate the principal components.

X = df_zscore.values  # shape: (n_samples, 5)

pca = PCA(n_components=5)  # keep all 5 components first to see full picture
pca.fit(X)

explained_var       = pca.explained_variance_ratio_
cumulative_var      = np.cumsum(explained_var)

pca_summary = pd.DataFrame({
    'PC': [f'PC{i+1}' for i in range(5)],
    'Explained Variance (%)': (explained_var * 100).round(2),
    'Cumulative Variance (%)': (cumulative_var * 100).round(2)
})

print('PCA Explained Variance Summary:')
print(pca_summary.to_string(index=False))

In [ ]:
# ── Scree plot & cumulative variance ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
axes[0].bar(pca_summary['PC'], explained_var * 100, color='steelblue', edgecolor='white')
axes[0].set_title('Scree Plot – Explained Variance per Component')
axes[0].set_ylabel('Explained Variance (%)')
axes[0].set_xlabel('Principal Component')

# Cumulative variance
axes[1].plot(pca_summary['PC'], cumulative_var * 100, marker='o', color='darkorange', linewidth=2)
axes[1].axhline(y=90, color='gray', linestyle='--', label='90% threshold')
axes[1].axhline(y=95, color='red', linestyle='--', label='95% threshold')
axes[1].set_title('Cumulative Explained Variance')
axes[1].set_ylabel('Cumulative Variance (%)')
axes[1].set_xlabel('Principal Component')
axes[1].legend()
axes[1].set_ylim([0, 105])

plt.tight_layout()
plt.show()

In [ ]:
# ── PCA loadings (component matrix) ────────────────────────────────────────
loadings = pd.DataFrame(
    pca.components_.T,
    index=NUM_COLS,
    columns=[f'PC{i+1}' for i in range(5)]
).round(3)

print('PCA Loadings (how much each original feature contributes to each PC):')
print(loadings.to_string())

In [ ]:
# ── Heatmap of loadings ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(loadings, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('PCA Loadings Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# ── Scatter plot: PC1 vs PC2 colored by Product Category ───────────────────
X_pca = pca.transform(X)

pca_df = pd.DataFrame(X_pca[:, :2], columns=['PC1', 'PC2'])
pca_df['Product_Category'] = cleaned['Product_Category'].values

fig, ax = plt.subplots(figsize=(10, 6))
for cat, grp in pca_df.groupby('Product_Category'):
    ax.scatter(grp['PC1'], grp['PC2'], label=cat, alpha=0.6, s=30)

ax.set_xlabel(f'PC1 ({explained_var[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({explained_var[1]*100:.1f}% variance)')
ax.set_title('PCA – PC1 vs PC2 (colored by Product Category)')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── How many components to keep? ────────────────────────────────────────────
n_90 = np.argmax(cumulative_var >= 0.90) + 1
n_95 = np.argmax(cumulative_var >= 0.95) + 1

print(f'Components needed to explain ≥ 90% of variance: {n_90}')
print(f'Components needed to explain ≥ 95% of variance: {n_95}')

print(f"""
PCA Interpretation:
──────────────────
• PC1 explains {explained_var[0]*100:.1f}% of the total variance.
  It is strongly influenced by Sales_Amount and Profit, suggesting it captures
  overall revenue and profitability.

• PC2 explains {explained_var[1]*100:.1f}% of the total variance.
  It contrasts Unit_Price with Units_Sold, capturing the trade-off between
  high-price/low-volume vs low-price/high-volume products.

• PC3 captures the remaining variance related to Discount behaviour.

• Together, PC1 and PC2 account for {cumulative_var[1]*100:.1f}% of total variance.
  Keeping {n_90} component(s) preserves ≥ 90% of information, making the reduced
  representation useful for downstream tasks like clustering or regression
  while reducing dimensionality.
""")

---
## Conclusion

| Task | Method Applied | Key Outcome |
|------|---------------|-------------|
| 1 – Data Quality Issues | Manual inspection + summary statistics | 7 categories of issues identified |
| 2 – Missing Values | Group-based Median Imputation | All NaN values filled; distribution preserved |
| 3 – Outlier Detection & Handling | IQR method + Winsorization (capping) | Outliers capped at bounds; no rows removed |
| 4 – Normalization | Min-Max Scaler + Z-score (StandardScaler) | Two normalized versions produced for comparison |
| 5 – PCA | PCA on Z-score data | PC1+PC2 capture most variance; loadings interpreted |